# Ancient Tamil (Palm-Leaf Manuscript) Character Recognition
## Classical Machine Learning Pipeline: SIFT + Gabor + PCA + Classifiers
### தமிழ் பனையோலை கையெழுத்துப் பிரதி எழுத்துணர்தல் (*Panaiyolai Suvadigal OCR*)

---

## 🚀 HOW TO RUN THIS PROJECT

### Option 1: Run Inside This Notebook (Interactive Walkthrough)
- **In VS Code / Jupyter Notebook:** Click **"Run All"** at the top of the editor window, or press **Shift + Enter** on each cell from top to bottom.
- All visual figures (preprocessed restorations, Gabor filters, scree plot, confusion matrix, error analysis, and benchmark charts) will render inline interactively.

### Option 2: Run Master Pipeline from Terminal (Command Line)
Open PowerShell or Command Prompt, navigate to this directory, and run:
```powershell
cd c:\Users\HARSHIT\Documents\ML\ancient_tamil_ocr
python main.py
```
*Executes dataset verification, preprocessing, Gabor+SIFT feature extraction, PCA compression, 10-fold cross validation, and generates all 300 DPI publication figures in `results/` in ~34 seconds on CPU.*

### Option 3: Run with Classical Neural Network (Shallow MLP)
```powershell
python main.py --classifier MLP
```

### Option 4: Test Any Single Character Image (`predict.py`)
```powershell
python predict.py --image data/01_a/01_a_sample_001.png
```
*Outputs predicted class, Devanagari/Tamil glyph, confidence score, top-3 candidates, and saves an annotated prediction card in `results/prediction_output.png`.*

---

### Pipeline Overview
1. **Preprocessing:** Grayscale $\rightarrow$ 64$\times$64 resizing $\rightarrow$ Gaussian denoising $\rightarrow$ Contrast Limited Adaptive Histogram Equalization (CLAHE) $\rightarrow$ Sauvola adaptive binarization (suited for degraded palm leaf manuscripts) $\rightarrow$ Morphological cleanup.
2. **Feature Extraction:**
   - **Gabor Filter Bank:** Multi-orientation and multi-frequency spatial filters capturing character stroke directions (mean, std, energy).
   - **Dense SIFT:** Scale-Invariant Feature Transform on a regular dense grid with Bag-of-Visual-Words (BoVW) spatial pooling.
   - **HOG:** Histogram of Oriented Gradients.
   - **Statistical Features:** Zoning density (4$\times$4), diagonal features, ink centroid, and horizontal/vertical projection profiles.
   - **Paper Combined Set:** Gabor + Dense SIFT.
3. **Dimensionality Reduction:** Principal Component Analysis (PCA) retaining ~95% cumulative explained variance (reducing 416-D down to 69-D).
4. **Classifiers:** Support Vector Machine (Polynomial, RBF, Linear kernels), k-Nearest Neighbors (k-NN), Random Forest, and Classical Shallow Multi-Layer Perceptron (MLP).
5. **Validation & Evaluation:** Stratified 10-fold cross-validation, held-out test evaluation, confusion matrix, error analysis, and component ablation study.

> **Strict Constraint:** This project uses **ONLY classical machine learning** (scikit-learn, OpenCV, scikit-image, NumPy, SciPy). Absolutely no deep learning frameworks (no CNNs, no PyTorch, no TensorFlow/Keras).

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure UTF-8 on Windows consoles
if sys.platform.startswith('win'):
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

# Configure inline plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = ['Nirmala UI', 'Mangal', 'DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# Import project modules
import config
import dataset
import preprocess
import features
import models
import evaluate
import predict

print('All modules successfully loaded!')

## 1. Dataset Loading & Palm-Leaf Manuscript Representation
The dataset loader scans data/<class_name>/*.png|jpg. If the directory is missing or empty, it automatically synthesizes realistic palm-leaf ancient handwritten character images simulating palm-leaf fibrous background grain, ink bleed/fading, and affine stroke variations.

### 📥 Dataset & Manuscript Download Links
- 📦 **[Download Complete Repo & Dataset (.ZIP)](https://github.com/harshit612007/ML_project_ocr/archive/refs/heads/main.zip)**
- 📂 **[Browse Character Dataset (data/)](https://github.com/harshit612007/ML_project_ocr/tree/main/ancient_tamil_ocr/data)**
- 📜 **[Browse Historical Manuscripts (manuscripts/)](https://github.com/harshit612007/ML_project_ocr/tree/main/ancient_tamil_ocr/manuscripts)**
- 🏛️ **Primary Scans:** [Tolkāppiyam (752 KB)](https://upload.wikimedia.org/wikipedia/commons/e/ec/Tolkappiyam_Palm_Leaf.jpg) | [Ciṟupañcamūlam (55 KB)](https://upload.wikimedia.org/wikipedia/commons/5/53/Cirupancamulam_Tamil_Palm_Leaf.jpg) | [Tamil Prayers (33 KB)](https://upload.wikimedia.org/wikipedia/commons/6/6f/Tamil_Palm_Leaf_Prayers.jpg)
- 🌐 **Digital Archives:** [Tamil Virtual Academy (TVA)](https://www.tamildigitallibrary.in/) | [French Institute of Pondicherry (IFP)](https://www.ifpindia.org/manuscripts/) | [British Library Endangered Archives (EAP314)](https://eap.bl.uk/project/EAP314)


In [ ]:
# Load dataset with stratified 80/20 train/test split (fixed random seed = 42)
X_train_raw, X_test_raw, y_train, y_test, class_names, file_paths = dataset.load_character_dataset(
    data_dir=config.DATA_DIR,
    test_size=config.TRAIN_TEST_SPLIT_RATIO,
    random_state=config.RANDOM_SEED,
    auto_generate_if_missing=True
)

print(f"Number of Character Classes: {len(class_names)}")
print(f"Training Samples (80%):    {len(X_train_raw)}")
print(f"Held-Out Test Samples (20%): {len(X_test_raw)}")
print("\nClasses and Tamil Glyphs:")
for c in class_names:
    glyph = config.TAMIL_GLYPH_MAP.get(c, '')
    print(f"  * {c:<10} -> {glyph}")

## 2. Preprocessing Pipeline
Historical palm-leaf manuscripts exhibit severe challenges:
- Aging ochre/brown leaf background with fibrous horizontal veins
- Non-uniform illumination and fading iron-gall/carbon ink
- Micro-cracks and stroke fractures

We address these issues through a 6-stage classical computer vision pipeline:
1. **Grayscale conversion**
2. **Resizing to 64$\times$64** (standardized resolution)
3. **Gaussian smoothing** to suppress high-frequency fibrous grain
4. **CLAHE** to enhance local contrast between faded ink and dark leaf veins
5. **Sauvola adaptive binarization** ($T = m(1 + k(\frac{s}{R} - 1))$)
6. **Morphological opening & closing** to eliminate speckles and heal hairline stroke breaks

In [ ]:
# Inspect intermediate stages for representative samples
fig, axes = plt.subplots(5, 5, figsize=(14, 13))
stage_keys = ['Original', 'Grayscale', 'CLAHE Enhanced', 'Sauvola Binarized', 'Morph Cleaned']
stage_titles = ['1. Original', '2. Grayscale (64x64)', '3. CLAHE', '4. Sauvola Binary', '5. Morph Cleaned']

for i in range(5):
    stages = preprocess.get_pipeline_stages(X_train_raw[i * 40])
    cls_label = class_names[y_train[i * 40]]
    glyph = config.TAMIL_GLYPH_MAP.get(cls_label, '')
    
    for j, key in enumerate(stage_keys):
        ax = axes[i, j]
        img = stages[key]
        if j == 0:
            ax.imshow(img)
            ax.set_ylabel(f"{cls_label} ({glyph})", fontsize=11, fontweight='bold', color='#2c3e50')
        else:
            ax.imshow(img, cmap='gray')
        if i == 0:
            ax.set_title(stage_titles[j], fontsize=11, fontweight='semibold', pad=8)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Ancient Tamil Palm-Leaf Preprocessing Walkthrough', fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout()
plt.show()

## 3. Preprocess Dataset Batches

In [ ]:
X_train_prep = preprocess.preprocess_dataset(X_train_raw)
X_test_prep = preprocess.preprocess_dataset(X_test_raw)
print(f"Preprocessed Train Shape: {X_train_prep.shape}")
print(f"Preprocessed Test Shape:  {X_test_prep.shape}")

## 4. Feature Extraction: Gabor Filter Bank + Dense SIFT
Following the core methodology of **Narang et al. (2020)**:
- **Gabor Filters:** 8 orientations ($0, \frac{\pi}{8}, \dots, \frac{7\pi}{8}$) $\times$ 4 spatial frequencies capturing directional frequency energy $\rightarrow$ 96 features.
- **Dense SIFT:** Regular dense keypoint grid across the 64$\times$64 character, pooled via Bag-of-Visual-Words (BoVW) and spatial pyramid quadrant pooling $\rightarrow$ 320 features.
- **Combined Set:** Concatenation yields 416 discriminative attributes, normalized via `StandardScaler` fitted strictly on `X_train`.

In [ ]:
# Fit feature pipeline strictly on training set to avoid data leakage
feat_pipeline = features.FeaturePipeline(feature_type="gabor_sift", random_state=config.RANDOM_SEED)
X_train_feat = feat_pipeline.fit_transform(X_train_prep)
X_test_feat = feat_pipeline.transform(X_test_prep)

raw_dim = X_train_feat.shape[1]
print(f"Raw Combined Feature Vector Length: {raw_dim} features per sample")

## 5. Dimensionality Reduction: Principal Component Analysis (PCA)
To eliminate multicollinearity and reduce training time while maintaining high recognition accuracy, PCA is fitted on the standardized training features to retain ~95% of cumulative explained variance.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=config.PCA_VARIANCE_RATIO, random_state=config.RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_feat)
X_test_pca = pca.transform(X_test_feat)

pca_dim = X_train_pca.shape[1]
var_pct = np.sum(pca.explained_variance_ratio_) * 100.0

print(f"Feature Length BEFORE PCA: {raw_dim}")
print(f"Feature Length AFTER PCA:  {pca_dim} (Dimensionality reduced by {((raw_dim - pca_dim) / raw_dim)*100:.1f}%)")
print(f"Cumulative Explained Variance Retained: {var_pct:.2f}%")

# Plot Cumulative Explained Variance Curve
cum_var = np.cumsum(pca.explained_variance_ratio_) * 100.0
plt.figure(figsize=(9, 5))
plt.plot(np.arange(1, len(cum_var) + 1), cum_var, color='#2980b9', lw=2.2, label='Cumulative Explained Variance')
plt.axhline(95.0, color='#e74c3c', ls='--', lw=1.8, label='95% Target Variance Threshold')
plt.axvline(pca_dim, color='#27ae60', ls=':', lw=1.8, label=f'Retained Dimensions: {pca_dim}')
plt.plot(pca_dim, cum_var[-1], 'o', color='#27ae60', markersize=8)
plt.title('PCA Cumulative Explained Variance vs. Number of Components', fontsize=13, fontweight='bold', pad=10)
plt.xlabel('Number of Principal Components', fontsize=11, fontweight='semibold')
plt.ylabel('Cumulative Variance Explained (%)', fontsize=11, fontweight='semibold')
plt.ylim(0, 105)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.show()

## 6. Stratified 10-Fold Cross-Validation
As conducted in Narang et al. (2020), the proposed classifier (Polynomial kernel SVM, degree=3) is evaluated across a 10-fold stratified cross-validation protocol.

In [ ]:
# Retrieve base Polynomial SVM (Paper winning classifier)
clf_poly = models.get_classifiers(random_state=config.RANDOM_SEED)["SVM-Poly"]

cv_results = models.run_tenfold_cross_validation(
    clf_poly,
    X_train_pca,
    y_train,
    cv_folds=10,
    random_state=config.RANDOM_SEED
)

print("="*55)
print(" STRATIFIED 10-FOLD CROSS-VALIDATION RESULTS (SVM-Poly)")
print("="*55)
print(f" Mean Accuracy:  {cv_results['mean_accuracy']*100:.2f}% (+/- {cv_results['std_accuracy']*100:.2f}%)")
print(f" Mean Macro F1:  {cv_results['mean_f1']*100:.2f}% (+/- {cv_results['std_f1']*100:.2f}%)")
print(f" Mean Precision: {cv_results['mean_precision']*100:.2f}% (+/- {cv_results['std_precision']*100:.2f}%)")
print(f" Mean Recall:    {cv_results['mean_recall']*100:.2f}% (+/- {cv_results['std_recall']*100:.2f}%)")
print("-"*55)
for fold_idx, (acc, f1) in enumerate(zip(cv_results['fold_accuracies'], cv_results['fold_f1s']), 1):
    print(f"   Fold {fold_idx:2d} -> Accuracy: {acc*100:5.2f}% | Macro F1: {f1*100:5.2f}%")
print("="*55)

## 7. Held-Out 20% Test Evaluation & Confusion Matrix

In [ ]:
# Fit model on full training set and evaluate on test set
clf_poly.fit(X_train_pca, y_train)
test_preds = clf_poly.predict(X_test_pca)

metrics = evaluate.evaluate_predictions(y_test, test_preds, class_names)
print(f"Test Accuracy:  {metrics['accuracy']*100:.2f}%")
print(f"Test Macro F1:  {metrics['f1_macro']*100:.2f}%")
print(f"Test Precision: {metrics['precision_macro']*100:.2f}%")
print(f"Test Recall:    {metrics['recall_macro']*100:.2f}%")
print("\nClassification Report:")
print(metrics["classification_report"])

# Confusion Matrix Visualization
cm = metrics['confusion_matrix']
cm_norm = cm.astype(np.float32) / np.maximum(1, np.sum(cm, axis=1, keepdims=True))
display_labels = [f"{c}\n({config.TAMIL_GLYPH_MAP.get(c, '')})" for c in class_names]

plt.figure(figsize=(9, 7))
annot = np.empty_like(cm, dtype=object)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        annot[i, j] = f"{cm[i, j]}\n({cm_norm[i, j]*100:.0f}%)" if cm[i, j] > 0 else "0"

sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', xticklabels=display_labels, yticklabels=display_labels,
            cbar_kws={'label': 'Normalized Recall'}, linewidths=0.5, linecolor='#e0e0e0')
plt.title('Confusion Matrix: Ancient Tamil Characters (SVM-Poly + Gabor/SIFT)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Predicted Class', fontsize=11, fontweight='semibold')
plt.ylabel('True Class', fontsize=11, fontweight='semibold')
plt.tight_layout()
plt.show()

## 8. Error Analysis: Most-Confused Ancient Tamil Pairs
Historical palm-leaf characters have structural stroke resemblances, which when combined with faded stylus strokes or leaf fiber erosion, lead to characteristic misclassifications.

In [ ]:
df_confused = evaluate.identify_most_confused_pairs(metrics['confusion_matrix'], class_names, top_n=5)
display(df_confused)

## 9. Comprehensive Feature Set $\times$ Classifier Benchmarking Matrix
Compare: Gabor, SIFT, Gabor+SIFT, HOG, Statistical, and All Combined $\times$ 6 Classical Classifiers (SVM-Poly, SVM-RBF, SVM-Linear, k-NN, Random Forest, MLP).

In [ ]:
results_csv = config.RESULTS_DIR / "results_table.csv"
if results_csv.exists():
    df_bench = pd.read_csv(results_csv)
    pivot_acc = df_bench.pivot(index="Feature_Set", columns="Classifier", values="Accuracy_Pct")
    
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot_acc, annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={'label': 'Test Accuracy (%)'},
                linewidths=0.8, linecolor='#ffffff')
    plt.title('Benchmark Matrix: Feature Set x Classifier Accuracy (%)', fontsize=13, fontweight='bold', pad=12)
    plt.xlabel('Classifier', fontsize=11, fontweight='semibold')
    plt.ylabel('Feature Extraction Technique', fontsize=11, fontweight='semibold')
    plt.tight_layout()
    plt.show()
    
    display(pivot_acc)
else:
    print("Run main.py first to generate results_table.csv")

## 10. Component Ablation Study
Quantifying the isolated contributions of:
1. **Full Proposed Pipeline:** Preprocessing (CLAHE + Sauvola) + Gabor/SIFT + PCA
2. **Without PCA:** Preprocessing + Gabor/SIFT + Full Vector (No PCA)
3. **Without Preprocessing:** Raw Grayscale Resized + Gabor/SIFT + PCA
4. **Baseline:** Raw Pixels + PCA

In [ ]:
ablation_csv = config.RESULTS_DIR / "ablation_study.csv"
if ablation_csv.exists():
    df_ablation = pd.read_csv(ablation_csv)
    display(df_ablation)
    
    plt.figure(figsize=(10, 5))
    bars = plt.bar(df_ablation["Condition"], df_ablation["Accuracy_Pct"], color=['#27ae60', '#2980b9', '#f39c12', '#e74c3c'],
                   edgecolor='#2c3e50', width=0.55)
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, h + 1.2, f"{h:.1f}%", ha='center', va='bottom', fontweight='bold')
    plt.ylim(0, 110)
    plt.ylabel('Test Accuracy (%)', fontsize=11, fontweight='semibold')
    plt.xticks(rotation=15, ha='right', fontsize=9, fontweight='semibold')
    plt.title('Ablation Analysis: Contribution of Preprocessing, SIFT+Gabor, and PCA', fontsize=13, fontweight='bold', pad=12)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("Run main.py first to generate ablation_study.csv")

## 11. Single-Image Inference Demonstration

In [ ]:
# Run inference on a sample character image
sample_test_path = list(config.DATA_DIR.glob("01_a/*.png"))[0]
best_class, best_conf, top_preds = predict.predict_single_character(
    image_path=str(sample_test_path),
    save_card=True
)

## 12. Authentic Historical Palm-Leaf Manuscript Processing (Tolkāppiyam & Ciṟupañcamūlam)
Here we test the complete classical pipeline on a **genuine historical palm-leaf manuscript** (*Panaiyolai Suvadigal*):
- Source 1: Authentic digital scan of the **Tolkāppiyam (தொல்காப்பியம்)** palm leaf (`manuscripts/tolkappiyam_palm_leaf.jpg`).
- Source 2: Classical **Ciṟupañcamūlam (சிறுபஞ்சமூலம்)** manuscript from the U.V. Swaminatha Iyer Library (`manuscripts/cirupancamulam_palm_leaf.jpg`).

The module restores the real leaf via Sauvola adaptive binarization, segments individual character contours along the horizontal strip, and classifies them using our trained classical ML models.

In [ ]:
import process_manuscript
from PIL import Image

# Process the genuine Tolkappiyam palm leaf manuscript
card_path, extracted_chars = process_manuscript.process_historical_manuscript(
    manuscript_path='manuscripts/tolkappiyam_palm_leaf.jpg',
    max_characters_to_extract=12
)

# Display the publication analysis figure inline
if card_path.exists():
    display(Image.open(card_path))
else:
    print('Manuscript card not found.')